# **Preparation Notebook**



---
## Setup Environment

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

---
## Student Information

In [ ]:
group_name = "Group 21"
student_name = "Vishwesh Kalyam"
student_id = "25424211"


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='group_name', value=group_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [ ]:
! pip install matplotlib seaborn scikit-learn -q


### 0.b Import Packages

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
import pandas as pd
import altair as alt
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler


---
## A. Feature Selection


## A.0 Load Data

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load datasets
try:
  customer_df = pd.read_csv(at.folder_path / "customer.csv")
  person_df = pd.read_csv(at.folder_path / "person.csv")
  product_category_df = pd.read_csv(at.folder_path / "product_category.csv")
  product_cost_history_df = pd.read_csv(at.folder_path / "product_cost_history.csv")
  product_list_price_history_df = pd.read_csv(at.folder_path / "product_list_price_history.csv")
  product_sub_category_df = pd.read_csv(at.folder_path / "product_sub_category.csv")
  product_df = pd.read_csv(at.folder_path / "product.csv")
  sales_order_detail_df = pd.read_csv(at.folder_path / "sales_order_detail.csv")
  sales_order_header_df = pd.read_csv(at.folder_path / "sales_order_header.csv")
  sales_territory_df = pd.read_csv(at.folder_path / "sales_territory.csv")
  special_offer_product_df = pd.read_csv(at.folder_path / "special_offer_product.csv")
  special_offer_df = pd.read_csv(at.folder_path / "special_offer.csv")
  store_df = pd.read_csv(at.folder_path / "store.csv")
  unit_measure_df = pd.read_csv(at.folder_path / "unit_measure.csv")
except Exception as e:
  print(e)

### A.1 Approach 1 — Domain Knowledge

In [ ]:
sales_order_header_df["order_date"] = pd.to_datetime(sales_order_header_df["order_date"])

header_sorted = sales_order_header_df.sort_values(["customer_id", "order_date"])
first_order = header_sorted.groupby("customer_id").first().reset_index()

order_counts = sales_order_header_df.groupby("customer_id").size().reset_index(name="total_orders")
first_order = first_order.merge(order_counts, on="customer_id")

first_order["repeat_buyer"] = (first_order["total_orders"] > 1).astype(int)
first_order["order_month"] = first_order["order_date"].dt.month

cust_simple = customer_df[["customer_id","store_id"]].drop_duplicates(subset="customer_id")
df = first_order.merge(cust_simple, on="customer_id", how="left")
df["is_store"] = df["store_id"].notna().astype(int)
df = df.drop(columns=["store_id"])

print(df.shape)
print(df["repeat_buyer"].value_counts())


In [ ]:
feature_selection_1_insights = """
The first approach uses domain knowledge to select features that should be known at the time of a customer first order - total_due (the value of their first order), online_order_flag (channel preference), is_store (B2B vs B2C indicator), territory_id (geographic region) and order_month (seasonal context). These were chosen because they reflect only information available at the moment of the first transaction, which is when a retention prediction would realistically be made in a real business setting.
"""


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_1_insights', value=feature_selection_1_insights)

### A.2 Approach 2 — Correlation Analysis

In [ ]:
le = LabelEncoder()
df["territory_enc"] = le.fit_transform(df["territory_id"])

num_cols = ["total_due", "online_order_flag", "is_store", "territory_enc", "order_month", "repeat_buyer"]
corr = df[num_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues")
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()


In [ ]:
feature_selection_2_insights = """
The correlation matrix shows that total_due has the strongest positive correlation with repeat_buyer, suggesting that higher first-order values are associated with returning customers. The other features show weaker but distinct relationships with the target, and no two features are so highly correlated with each other that one would need to be removed.
"""


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_2_insights', value=feature_selection_2_insights)

### A.n Additional Approach

In [ ]:
feature_selection_n_insights = """
No additional feature selection approaches were required - the domain knowledge selection and correlation analysis together provided sufficient justification for the final feature set.
"""


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_n_insights', value=feature_selection_n_insights)

### A.z Final Selection of Features

In [ ]:
df = df.drop(columns=["territory_id"])

features_list = ["total_due", "online_order_flag", "is_store", "territory_enc", "order_month"]
target = "repeat_buyer"

print("Features:", features_list)
print("Target:", target)
print("Dataset shape:", df.shape)


In [ ]:
feature_selection_explanations = """
The final feature set consists of five variables - total_due, online_order_flag, is_store, territory_enc and order_month - all derived from the customer first order, deliberately avoiding any features that depend on subsequent purchase history to prevent target leakage. This represents a realistic prediction scenario where a retention model is applied immediately after the first transaction.
"""


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## B. Data Cleaning

### B.1 Fixing Duplicate Rows

In [ ]:
print("Duplicates before:", df.duplicated(subset="customer_id").sum())
df = df.drop_duplicates(subset="customer_id")
print("Shape after dedup:", df.shape)


In [ ]:
data_cleaning_1_explanations = """
The customer dataset contained 3,944 duplicate rows as identified during EDA, so the dataset was deduplicated on customer_id to ensure each customer appears only once in the final modelling dataset.
"""


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### B.2 Fixing Missing territory_enc Values

In [ ]:
print("Missing territory before:", df["territory_enc"].isna().sum())
df = df.dropna(subset=["territory_enc"])
print("Shape after dropping missing:", df.shape)


In [ ]:
data_cleaning_2_explanations = """
A small number of records had no territory_id assigned, and since this information cannot be reliably imputed without introducing bias, the affected rows were dropped from the dataset.
"""


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### B.3 Verifying No Remaining Nulls

In [ ]:
print("Remaining nulls in features and target:")
print(df[features_list + [target]].isnull().sum())


In [ ]:
data_cleaning_3_explanations = """
After deduplication and dropping rows with missing territory values, the dataset has no remaining null values across the features or the target variable, confirming it is ready for splitting and modelling.
"""


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

### B.n Additional Cleaning

In [ ]:
data_cleaning_n_explanations = """
No further cleaning was required.
"""


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_n_explanations', value=data_cleaning_n_explanations)

---
## C. Split Datasets


In [ ]:
X = df[features_list]
y = df[target]

X_train_raw, X_temp, y_train_raw, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val_raw, X_test_raw, y_val_raw, y_test_raw = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

training_df = X_train_raw.copy()
training_df[target] = y_train_raw.values

validation_df = X_val_raw.copy()
validation_df[target] = y_val_raw.values

testing_df = X_test_raw.copy()
testing_df[target] = y_test_raw.values

X_train = X_train_raw.copy()
y_train = y_train_raw.to_frame()

X_val = X_val_raw.copy()
y_val = y_val_raw.to_frame()

X_test = X_test_raw.copy()
y_test = y_test_raw.to_frame()

y_train.to_csv(at.folder_path / "y_train.csv", index=False)
y_val.to_csv(at.folder_path / "y_val.csv", index=False)
y_test.to_csv(at.folder_path / "y_test.csv", index=False)

print(f"Train: {training_df.shape}, Val: {validation_df.shape}, Test: {testing_df.shape}")
print("Target distribution in train:")
print(training_df[target].value_counts())


In [ ]:
data_splitting_explanations = """
A random 70/15/15 train-validation-test split was used with stratification on the target variable to ensure the class balance (roughly 60/40 between one-time and repeat buyers) is preserved across all three sets. A fixed random_state of 42 is used to make the split reproducible. The test set is held out entirely and only used for final model evaluation.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_splitting_explanations', value=data_splitting_explanations)

---
## D. Feature Engineering

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  training_df_eng = training_df.copy()
  validation_df_eng = validation_df.copy()
  testing_df_eng = testing_df.copy()
except Exception as e:
  print(e)

### D.1 New Feature `log_total_due`

In [ ]:
for d in [training_df_eng, validation_df_eng, testing_df_eng]:
    d["log_total_due"] = np.log1p(d["total_due"])

print(training_df_eng[["total_due","log_total_due"]].describe())


In [ ]:
feature_engineering_1_explanations = """
A log transformation was applied to total_due to address the heavy right skew observed in the EDA - first-order values range from $1.50 to nearly $200,000, and without transformation a small number of very high-value orders would disproportionately influence distance-based models. log1p is used rather than log to safely handle any zero values.
"""


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### D.2 New Feature — Retained for Future Use

In [ ]:
print("No additional engineered feature added — log_total_due was the only transformation needed.")


In [ ]:
feature_engineering_2_explanations = """
Beyond the log transformation, no additional engineered features were created as the original five features combined with log_total_due provide sufficient predictive signal without overcomplicating the feature space.
"""


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### D.3 Additional Feature Engineering

In [ ]:
feature_engineering_3_explanations = """
No further feature engineering was performed.
"""


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_3_explanations', value=feature_engineering_3_explanations)

### D.n Additional Feature Engineering

In [ ]:
feature_engineering_n_explanations = """
No additional feature engineering steps were needed.
"""


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_n_explanations', value=feature_engineering_n_explanations)

---
## E. Data Preparation for Modeling

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  X_train = training_df_eng.copy()
  X_val = validation_df_eng.copy()
  X_test = testing_df_eng.copy()
except Exception as e:
  print(e)

### E.1 Data Transformation — Standard Scaling

In [ ]:
scale_features = ["total_due", "log_total_due"]

scaler = StandardScaler()
X_train[scale_features] = scaler.fit_transform(X_train[scale_features])
X_val[scale_features]   = scaler.transform(X_val[scale_features])
X_test[scale_features]  = scaler.transform(X_test[scale_features])

print("Scaling applied.")
print(X_train[scale_features].describe())


In [ ]:
data_transformation_1_explanations = """
StandardScaler was applied to the continuous numeric features - total_due and log_total_due. The scaler is fit only on the training set and then applied to validation and test sets to prevent data leakage. Scaling is particularly important for the SVM and KNN classifiers which are sensitive to feature magnitude.
"""


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### E.2 Data Transformation — Label Encoding Check

In [ ]:
print("territory_enc unique values:", sorted(X_train["territory_enc"].unique()))
print("online_order_flag unique values:", sorted(X_train["online_order_flag"].unique()))
print("is_store unique values:", sorted(X_train["is_store"].unique()))


In [ ]:
data_transformation_2_explanations = """
territory_id was label encoded into territory_enc using LabelEncoder during the feature selection stage. online_order_flag and is_store are already binary integers requiring no further encoding. All categorical features are now numeric and suitable for sklearn models.
"""


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_2_explanations', value=data_transformation_2_explanations)

### E.3 Final Dataset Check

In [ ]:
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)
print("Nulls in X_train:", X_train.isnull().sum().sum())
print("Features:", list(X_train.columns))


In [ ]:
data_transformation_3_explanations = """
The final prepared datasets contain no null values and all features are in numeric format. X_train, X_val and X_test are ready to be used directly in the baseline and classification notebooks.
"""


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_3_explanations', value=data_transformation_3_explanations)

### E.n Additional Transformations

In [ ]:
data_transformation_n_explanations = """
No further transformations were needed.
"""


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_n_explanations', value=data_transformation_n_explanations)

---
## F. Save Datasets

> Do not change this code

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL

try:
  X_train.to_csv(at.folder_path / 'X_train.csv', index=False)

  X_val.to_csv(at.folder_path / 'X_val.csv', index=False)

  X_test.to_csv(at.folder_path / 'X_test.csv', index=False)
except Exception as e:
  print(e)